# Chapter 5: Clustering — Finding Groups Without Labels

Clustering is **unsupervised learning**: no correct answers given.

The model discovers **natural groups** in the data by itself.

Use cases:
- Customer segmentation (who are our different types of customers?)
- Document grouping (which articles are about similar topics?)
- Anomaly detection (which data points don't belong to any group?)

---
## K-Means: The Most Common Clustering Algorithm

K-Means groups data into **K clusters** by finding K center points.

Algorithm:
1. **Place** K random center points
2. **Assign** each data point to its nearest center
3. **Move** each center to the average of its assigned points
4. **Repeat** steps 2-3 until centers stop moving

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

np.random.seed(42)

# Generate 3 groups of customers
# [annual_spending ($K), visit_frequency (per month)]
group_a = np.random.normal([10, 15], [3, 3], (20, 2))   # budget frequent shoppers
group_b = np.random.normal([50, 3], [5, 1], (20, 2))    # premium rare shoppers
group_c = np.random.normal([30, 8], [4, 2], (20, 2))    # mid-range regular shoppers

customers = np.vstack([group_a, group_b, group_c])

print(f"We have {len(customers)} customers.")
print(f"Features: annual spending ($K) and visits per month")
print(f"We DON'T know which group they belong to.")
print(f"\nFirst 5 customers:")
print(f"{'Spending ($K)':>14} {'Visits/month':>14}")
for c in customers[:5]:
    print(f"{c[0]:>14.1f} {c[1]:>14.1f}")

In [ ]:
# Run K-Means with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(customers)
centers = kmeans.cluster_centers_

print("K-Means found 3 groups:\n")
for i in range(3):
    cluster_data = customers[labels == i]
    print(f"Cluster {i}: {len(cluster_data)} customers")
    print(f"  Avg spending: ${cluster_data[:, 0].mean():.1f}K")
    print(f"  Avg visits:   {cluster_data[:, 1].mean():.1f}/month")
    print(f"  Center:       ({centers[i][0]:.1f}, {centers[i][1]:.1f})")
    print()

In [ ]:
# Visualize clusters as ASCII scatter plot
def ascii_scatter(data, labels, centers, width=60, height=20):
    markers = ['·', 'o', 'x']
    x_min, x_max = data[:, 0].min() - 2, data[:, 0].max() + 2
    y_min, y_max = data[:, 1].min() - 2, data[:, 1].max() + 2
    
    grid = [[' ' for _ in range(width)] for _ in range(height)]
    
    for point, label in zip(data, labels):
        col = int((point[0] - x_min) / (x_max - x_min) * (width - 1))
        row = int((1 - (point[1] - y_min) / (y_max - y_min)) * (height - 1))
        row = max(0, min(height - 1, row))
        col = max(0, min(width - 1, col))
        grid[row][col] = markers[label]
    
    for center in centers:
        col = int((center[0] - x_min) / (x_max - x_min) * (width - 1))
        row = int((1 - (center[1] - y_min) / (y_max - y_min)) * (height - 1))
        row = max(0, min(height - 1, row))
        col = max(0, min(width - 1, col))
        grid[row][col] = '★'
    
    print(f"  Visits/mo ↑")
    for row in grid:
        print(f"  │{''.join(row)}│")
    print(f"  └{'─' * width}┘")
    print(f"   Spending ($K) →")
    print(f"\n  · = Cluster 0   o = Cluster 1   x = Cluster 2   ★ = Center")

ascii_scatter(customers, labels, centers)

---
## Step by Step: Watching K-Means Converge

Let's manually run the algorithm to see how centers move.

In [ ]:
# Manual K-Means to show the iterations
np.random.seed(0)
K = 3
centers = customers[np.random.choice(len(customers), K, replace=False)]

print("Watching K-Means iterate:\n")

for iteration in range(6):
    # Step 1: Assign each point to nearest center
    distances = np.array([[np.sqrt(np.sum((p - c)**2)) for c in centers] for p in customers])
    labels = distances.argmin(axis=1)
    
    # Step 2: Move centers to mean of assigned points
    new_centers = np.array([customers[labels == k].mean(axis=0) for k in range(K)])
    
    # How much did centers move?
    movement = np.sqrt(np.sum((new_centers - centers)**2))
    
    print(f"Iteration {iteration + 1}:")
    for k in range(K):
        count = np.sum(labels == k)
        print(f"  Center {k}: ({centers[k][0]:5.1f}, {centers[k][1]:5.1f}) → ({new_centers[k][0]:5.1f}, {new_centers[k][1]:5.1f})  [{count} points]")
    print(f"  Total movement: {movement:.2f}")
    
    if movement < 0.01:
        print("  ★ Converged! Centers stopped moving.")
        break
    
    centers = new_centers
    print()

---
## Choosing K: How Many Clusters?

K-Means needs you to specify K. But how do you know the right number?

### Elbow Method
Try different K values, plot the total distance (inertia). Look for the "elbow" — where adding more clusters stops helping much.

In [ ]:
# Elbow method
print("Elbow Method: Finding the right K\n")
print(f"{'K':>3} {'Inertia':>12} {'Chart':>30}")
print(f"{'─'*3} {'─'*12} {'─'*30}")

inertias = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(customers)
    inertias.append(km.inertia_)
    bar_len = int(km.inertia_ / max(inertias) * 30)
    bar = "█" * bar_len
    elbow = " ← elbow (best K)" if k == 3 else ""
    print(f"{k:>3} {km.inertia_:>12.0f} {bar}{elbow}")

print("\nAfter K=3, inertia drops slowly → 3 is the right number.")
print("This matches our data: we generated 3 groups!")

---
## Silhouette Score: How Well-Separated Are Clusters?

Measures how similar each point is to its OWN cluster vs the NEAREST other cluster.

- **+1**: perfectly clustered
- **0**: on the boundary between clusters
- **-1**: probably in the wrong cluster

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

print("Silhouette Score for Different K:\n")
print(f"{'K':>3} {'Score':>8} {'Quality':>30}")
print(f"{'─'*3} {'─'*8} {'─'*30}")

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(customers)
    score = silhouette_score(customers, labels)
    bar = "█" * int(score * 30)
    best = " ← best" if k == 3 else ""
    print(f"{k:>3} {score:>8.3f}  {bar}{best}")

print("\nHighest silhouette = best separation between clusters.")

---
## Other Clustering Methods

K-Means assumes **round** clusters of similar size. Real data is messier.

In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering

print("Clustering Methods Comparison:\n")
print(f"{'Method':<25} {'Clusters':>8} {'Sil. Score':>10}")
print(f"{'─'*25} {'─'*8} {'─'*10}")

methods = {
    "K-Means (K=3)": KMeans(n_clusters=3, random_state=42, n_init=10),
    "Agglomerative (K=3)": AgglomerativeClustering(n_clusters=3),
    "DBSCAN (auto K)": DBSCAN(eps=5, min_samples=3),
}

for name, method in methods.items():
    labels = method.fit_predict(customers)
    n_clusters = len(set(labels) - {-1})
    if n_clusters >= 2:
        score = silhouette_score(customers, labels)
        print(f"{name:<25} {n_clusters:>8} {score:>10.3f}")
    else:
        print(f"{name:<25} {n_clusters:>8} {'N/A':>10}")

print("\nKey differences:")
print("  K-Means:        fast, needs K, assumes round clusters")
print("  Agglomerative:  builds tree of clusters, needs K")
print("  DBSCAN:         finds K automatically, handles weird shapes")

---
## Practical Example: Customer Segmentation

Let's put it all together with a realistic scenario.

In [ ]:
from sklearn.preprocessing import StandardScaler

# More realistic customer data
np.random.seed(42)
n = 100
customer_data = np.column_stack([
    np.concatenate([np.random.normal(25, 5, 40), np.random.normal(45, 8, 30), np.random.normal(35, 5, 30)]),  # age
    np.concatenate([np.random.normal(200, 50, 40), np.random.normal(800, 150, 30), np.random.normal(400, 80, 30)]),  # spending
    np.concatenate([np.random.normal(20, 5, 40), np.random.normal(4, 2, 30), np.random.normal(10, 3, 30)]),  # visits
])

# Scale features (important for K-Means!)
scaler = StandardScaler()
scaled = scaler.fit_transform(customer_data)

# Cluster
km = KMeans(n_clusters=3, random_state=42, n_init=10)
segments = km.fit_predict(scaled)

print("Customer Segmentation Results:\n")
segment_names = ["Budget Frequent", "Premium Selective", "Mid-Range Regular"]

for i in range(3):
    mask = segments == i
    data = customer_data[mask]
    print(f"Segment {i}: {segment_names[i]} ({mask.sum()} customers)")
    print(f"  Avg age:      {data[:, 0].mean():.0f} years")
    print(f"  Avg spending: ${data[:, 1].mean():.0f}/year")
    print(f"  Avg visits:   {data[:, 2].mean():.0f}/month")
    print()

print("Business actions:")
print("  Budget Frequent  → loyalty discounts, bulk deals")
print("  Premium Selective → VIP experience, exclusive products")
print("  Mid-Range Regular → upsell opportunities, personalized emails")

---
## Important: Scaling Matters!

K-Means uses **distance**. If one feature is in thousands and another is 0-1, the big feature dominates.

**Always scale your data before clustering.**

In [ ]:
# Demonstrate why scaling matters
km_unscaled = KMeans(n_clusters=3, random_state=42, n_init=10)
km_scaled = KMeans(n_clusters=3, random_state=42, n_init=10)

labels_unscaled = km_unscaled.fit_predict(customer_data)  # raw data
labels_scaled = km_scaled.fit_predict(scaled)              # scaled data

score_unscaled = silhouette_score(customer_data, labels_unscaled)
score_scaled = silhouette_score(scaled, labels_scaled)

print("Effect of Scaling:")
print(f"  Without scaling: silhouette = {score_unscaled:.3f}")
print(f"  With scaling:    silhouette = {score_scaled:.3f}")
print()
print("Why?")
print(f"  Age range:      {customer_data[:, 0].min():.0f} - {customer_data[:, 0].max():.0f}")
print(f"  Spending range: {customer_data[:, 1].min():.0f} - {customer_data[:, 1].max():.0f}")
print(f"  Visits range:   {customer_data[:, 2].min():.0f} - {customer_data[:, 2].max():.0f}")
print()
print("Without scaling, spending (range ~1000) drowns out")
print("visits (range ~25). StandardScaler makes all features equal.")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Clustering** | Unsupervised — finds groups without labels |
| **K-Means** | Place K centers, assign points, move centers, repeat |
| **Elbow Method** | Plot inertia vs K, pick the "elbow" |
| **Silhouette Score** | Measures cluster separation quality (-1 to +1) |
| **DBSCAN** | Finds K automatically, handles irregular shapes |
| **Scaling** | Always scale before clustering! |

---

### End of Part 1: Machine Learning

You now understand:
- What ML is and how it differs from traditional programming
- Regression (predict numbers) and Classification (predict categories)
- Tree-based models and ensemble methods
- Clustering (unsupervised grouping)

**Next: Part 2 — Neural Networks** (how deep learning builds on these foundations)